In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import tqdm
from torch.utils.tensorboard import SummaryWriter

import torch.optim as optim
import optuna

In [40]:
df_train = pd.read_csv(('../data/train_multilabel.csv'))
df_val = pd.read_csv(('../data/test_multilabel.csv'))
df_test = pd.read_csv(('../data/valid_multilabel.csv'))


In [41]:
label_cols = df_train.columns[6:].tolist()

In [42]:
#데이터 셋 만들기(멀티라벨 용으로)

class ComplainDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.label_cols = label_cols
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx,'text']
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        label = self.df.loc[idx, self.label_cols].values.astype("float32")

        item = {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32) 
        }
        return item

In [43]:
# 한국어 전용 토크나이져
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")

In [44]:
#dataset 만들기
train_dataset = ComplainDataset(df_train, tokenizer, label_cols)
val_dataset = ComplainDataset(df_val, tokenizer, label_cols)
test_dataset = ComplainDataset(df_test, tokenizer, label_cols)

In [45]:
#data_loaders 만들기
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [50]:
# 모델 생성
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/roberta-base",
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)

for params in model.parameters():
    params.requires_grad = False

for param in model.roberta.encoder.layer[11].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True
    
for name,module in model.named_parameters():
    print(name, module.requires_grad)

model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1010.68it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Co

roberta.embeddings.word_embeddings.weight False
roberta.embeddings.token_type_embeddings.weight False
roberta.embeddings.LayerNorm.weight False
roberta.embeddings.LayerNorm.bias False
roberta.embeddings.position_embeddings.weight False
roberta.encoder.layer.0.attention.self.query.weight False
roberta.encoder.layer.0.attention.self.query.bias False
roberta.encoder.layer.0.attention.self.key.weight False
roberta.encoder.layer.0.attention.self.key.bias False
roberta.encoder.layer.0.attention.self.value.weight False
roberta.encoder.layer.0.attention.self.value.bias False
roberta.encoder.layer.0.attention.output.dense.weight False
roberta.encoder.layer.0.attention.output.dense.bias False
roberta.encoder.layer.0.attention.output.LayerNorm.weight False
roberta.encoder.layer.0.attention.output.LayerNorm.bias False
roberta.encoder.layer.0.intermediate.dense.weight False
roberta.encoder.layer.0.intermediate.dense.bias False
roberta.encoder.layer.0.output.dense.weight False
roberta.encoder.layer.

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [36]:
def objective(trial):

    batch_size = trial.suggest_categorical('batch_size', [4,6,8,10,12,14,16])
    lr = trial.suggest_float('lr', 1e-7, 1e-4)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.to(device)
    epochs = 20
    total_loss = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0.0
        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()

        train_loss_sum /= len(train_loader.dataset)
        model.eval()
        val_loss_sum = 0.0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits

                loss = criterion(logits, labels)
                val_loss_sum += loss.item()    

                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()

        total_loss = val_loss_sum / len(val_loader.dataset)

        trial.report(total_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return total_loss
    
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

[I 2026-03-09 16:45:37,355] A new study created in memory with name: no-name-5d118ce2-03c5-4bb8-8005-3723f4f80937
[I 2026-03-09 16:57:13,424] Trial 0 finished with value: 0.6681749933347927 and parameters: {'batch_size': 16, 'lr': 1.654242164586801e-05}. Best is trial 0 with value: 0.6681749933347927.
[I 2026-03-09 17:10:45,713] Trial 1 finished with value: 1.0921400295467827 and parameters: {'batch_size': 10, 'lr': 4.606739327582081e-06}. Best is trial 0 with value: 0.6681749933347927.
[I 2026-03-09 17:24:38,747] Trial 2 finished with value: 1.1658789011437123 and parameters: {'batch_size': 10, 'lr': 8.623977915921695e-05}. Best is trial 0 with value: 0.6681749933347927.
[I 2026-03-09 17:37:48,961] Trial 3 finished with value: 1.174954774811512 and parameters: {'batch_size': 10, 'lr': 5.131618552772844e-05}. Best is trial 0 with value: 0.6681749933347927.
[I 2026-03-09 17:50:24,727] Trial 4 finished with value: 0.8417639724851593 and parameters: {'batch_size': 14, 'lr': 9.018513607432

In [37]:
print(study.best_trial.params)

{'batch_size': 16, 'lr': 1.654242164586801e-05}


In [52]:
writer = SummaryWriter()
optimizer = torch.optim.Adam(model.parameters(), lr=1.654242164586801e-05)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

model.to(device)

best_val_loss = 100000000
epochs = 60
early_stop_epochs = 5
early_stop_counter = 0
count = 0

for epoch in range(epochs):
    train_tqdm = tqdm.tqdm(train_loader)
    model.train()
    train_loss_sum = 0

    for batch in train_tqdm:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()

        writer.add_scalar("Loss/train_step", loss.item(), count)
        count += 1

        train_tqdm.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)
    print("avg_train_loss",avg_train_loss)

    model.eval()    
    all_preds = []
    all_labels = []
    val_loss_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_loss_sum += loss.item()    

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

        avg_val_loss = val_loss_sum / len(val_loader)
    print( "avg_val_loss",avg_val_loss, "epoch:", epoch)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "multiLabel_best_model.pt")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

        if early_stop_counter >= early_stop_epochs:
            print("Early stopping triggered.")
            break
    



100%|██████████| 253/253 [00:17<00:00, 14.17it/s, loss=0.1389]


avg_train_loss 0.17641353165443707
avg_val_loss 0.09934071833267807 epoch: 0


100%|██████████| 253/253 [00:17<00:00, 14.23it/s, loss=0.1325]


avg_train_loss 0.1306564459805432
avg_val_loss 0.0920253274962306 epoch: 1


100%|██████████| 253/253 [00:18<00:00, 14.02it/s, loss=0.1240]


...

avg_train_loss 0.01146669085450441
avg_val_loss 0.0444002702832222 epoch: 18


100%|██████████| 253/253 [00:18<00:00, 13.90it/s, loss=0.0094]


avg_train_loss 0.00992124214429747
avg_val_loss 0.045425736217293886 epoch: 19


100%|██████████| 253/253 [00:18<00:00, 13.99it/s, loss=0.0075]


avg_train_loss 0.008628785542098312
avg_val_loss 0.04732942347181961 epoch: 20
Early stopping triggered.


In [48]:
model.eval()

test_loss_sum = 0
all_preds = []
all_labels = []
test_tqdm = tqdm.tqdm(test_loader, desc="Test")
with torch.no_grad():
    for batch in test_tqdm:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        test_loss_sum += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        test_tqdm.set_postfix(loss=f"{loss.item():.4f}")
avg_test_loss = test_loss_sum / len(test_loader)

avg_test_loss

Test: 100%|██████████| 64/64 [00:04<00:00, 16.00it/s, loss=0.0126]


0.01119223388377577

In [32]:
text = "확정일자 받고싶은데 필요한게 머야?"

model.eval()

encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
pred_labels = []
for col, prob in zip(label_cols, probs):
    if prob >= 0.3:
        pred_labels.append((col, float(prob)))
pred_labels = sorted(pred_labels, key=lambda x: x[1], reverse=True)

print(pred_labels)
print(probs)

[('기관:읍·면·동 행정복지센터', 0.97264164686203), ('부서:민원행정팀', 0.9108037352561951), ('문서:신분증', 0.8662473559379578), ('민원명:확정일자 신청', 0.7930847406387329), ('문서:확정일자 신청서', 0.7372775077819824), ('문서:임대차계약서 원본', 0.7081646919250488)]
[8.4762536e-03 7.3263389e-03 1.3519323e-02 2.0364579e-03 3.5900667e-03
 1.5220273e-03 2.4705103e-03 1.8457631e-02 9.8655829e-03 7.2868243e-03
 4.4995584e-03 9.7264165e-01 1.4186603e-02 8.5847182e-03 2.1494839e-02
 8.0411676e-03 2.1249067e-02 4.1445643e-03 5.1518749e-03 5.6669493e-03
 2.3252899e-03 1.4492990e-03 6.5039866e-02 3.6643595e-03 5.1549673e-03
 1.3410510e-02 8.2912380e-03 4.0967707e-03 6.3438192e-03 1.0018479e-02
 4.8978049e-03 9.6279494e-03 5.7077529e-03 5.3815907e-03 6.0085948e-03
 3.7133782e-03 1.7591134e-02 3.2549191e-02 2.0045587e-03 2.3745187e-03
 2.1205412e-03 3.2689748e-03 6.0117561e-03 3.4842123e-03 4.3718321e-03
 3.5997201e-03 7.2957883e-03 7.1166055e-03 5.0503309e-03 2.5523581e-02
 1.0404636e-01 4.0870835e-03 5.2048550e-03 5.6291730e-03 2.4902369e-03
 